In [38]:
import warnings
warnings.filterwarnings('ignore')

import re

import string

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import missingno as msno

from mlxtend.evaluate import bias_variance_decomp
from mlxtend.feature_selection import SequentialFeatureSelector

from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.metrics import classification_report, roc_auc_score, f1_score, mean_absolute_percentage_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from statsmodels.stats.weightstats import ttest_ind

import nltk
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet, stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, classification_report
from catboost import CatBoostClassifier

In [39]:
def metrics_report(y_true, y_pred):
    """
    Выводит отчёт с основными метриками качества регрессии.
    Округляет до 4-х знаков после запятой и выводит значения R2 (коэффициент детерминации), RMSE (среднеквадратичная ошибка) и MAPE (средняя абсолютная процентная ошибка) для оценки качества предсказаний.

    Аргументы:
        y_true (numpy.ndarray): Истинные значения целевой переменной.
        y_pred (numpy.ndarray): Предсказанные значения целевой переменной.
    """
    print(f'R2 score: {r2_score(y_true, y_pred):.4f}')
    print(f'RMSE: {mean_squared_error(y_true, y_pred)**0.5:.4f}')
    print(f'MAPE: {mean_absolute_percentage_error(y_true, y_pred):.4f}')

In [55]:
import csv
import pandas as pd

path = '../women-clothing-accessories.3-class.balanced.csv'


with open(path, 'r', encoding='utf-8', errors='replace') as f:
    sample = f.read(5000)

dialect = csv.Sniffer().sniff(sample, delimiters=[',', ';', '\t', '|'])
print('delimiter:', repr(dialect.delimiter))

df_wc = pd.read_csv(
    path,
    sep=dialect.delimiter,
    encoding='utf-8',
    engine='python'
)

print(df_wc.shape)
print(df_wc.columns.tolist())
df_wc.head()


delimiter: '\t'
(90000, 2)
['review', 'sentiment']


,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


In [56]:
def regex_clean(text):
    """
    Очищает текст от ссылок

    Аргументы:
        text (str): Входной текст для очистки.

    Возвращает:
        str: Очищенный текст.
    """
    # Замена всех ссылок в тексте на пробел
    text = re.sub('http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', ' ', text)

    # Удаление всех цифр, спецсимволов и знаков пунктуации
    # Удаление лишних пробелов
    text = re.sub('\s+', ' ', text)
    
    return text

In [57]:



# ---------------------------
# Словари
# ---------------------------

POS_WORDS = {
    'хороший', 'отличный', 'супер', 'прекрасный', 'понравился', 'понравилось',
    'красивый', 'удобный', 'качественный', 'рекомендую', 'идеально', 'люблю',
    'замечательный', 'классный', 'мягкий', 'приятный'
}

NEG_WORDS = {
    'плохой', 'ужасный', 'отвратительный', 'кошмар', 'брак', 'некачественный',
    'разочарование', 'ужас', 'плохо', 'обман', 'тонкий', 'кривой',
    'неудобный', 'грязный', 'порванный', 'дешевый'
}

INTENSIFIERS = {
    'очень', 'сильно', 'крайне', 'совсем', 'слишком', 'безумно',
    'максимально', 'весьма', 'очень-очень'
}

NEGATIONS = {'не', 'нет', 'никогда', 'ни', 'нельзя'}

DELIVERY_WORDS = {
    'доставка', 'доставили', 'пришел', 'пришла', 'задержка', 'вернули',
    'посылка', 'курьер', 'приход', 'отправка', 'доставку'
}

QUALITY_WORDS = {
    'ткань', 'шов', 'материал', 'качество', 'нитки', 'синтетика',
    'брак', 'рисунок', 'криво', 'пошив', 'подкладка'
}

SIZE_WORDS = {
    'размер', 'маломерит', 'большемерит', 'узкий', 'широкий',
    'маленький', 'большой', 'короткий', 'длинный'
}

PATTERNS = {
    'has_not_recommend': r'не\s+рекомендую',
    'has_recommend': r'\bрекомендую\b',
    'has_liked': r'очень\s+понрав|понравил',
    'has_not_arrived': r'не\s+пришел|не\s+пришла|не\s+дошел|не\s+дошла',
    'has_refund': r'деньги\s+вернули|возврат|вернули\s+деньги',
    'has_mismatch': r'не\s+соответствует|не\s+такой|как\s+на\s+фото|не\s+как\s+на\s+фото',
    'has_bad_quality': r'плохое\s+качество|ужасн\w+\s+ткан\w+|торчат\s+нитки|кривой\s+шов',
    'has_good_quality': r'отличн\w+\s+качеств\w+|приятн\w+\s+ткан\w+|хорош\w+\s+качеств\w+',
    'has_size_issue': r'маломерит|большемерит|не\s+подошел\s+размер|не\s+подошла\s+по\s+размеру',
    'has_strong_negative_phrase': r'полное\s+разочарование|это\s+ужас|просто\s+ужас|кошмар',
    'has_strong_positive_phrase': r'в\s+восторге|очень\s+довольна|очень\s+доволен|просто\s+супер'
}

# ---------------------------
# Регексы
# ---------------------------

WORD_RE = re.compile(r'\b\w+\b', flags=re.UNICODE)
SENTENCE_SPLIT_RE = re.compile(r'[.!?]+')
MULTI_PUNCT_RE = re.compile(r'(\!\!+|\?\?+|\?\!+|\!\?+)')
REPEAT_CHAR_RE = re.compile(r'(.)\1{2,}', flags=re.UNICODE)
CAPS_WORD_RE = re.compile(r'\b[А-ЯЁA-Z]{2,}\b')
UPPERCASE_LETTER_RE = re.compile(r'[А-ЯЁA-Z]')
DIGIT_RE = re.compile(r'\d')
NEWLINE_RE = re.compile(r'[\r\n\t]+')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
MULTISPACE_RE = re.compile(r'\s+')

PUNCT_CHARS = set(string.punctuation + '«»—…„“”№')

# ---------------------------
# Нормализация
# ---------------------------

def normalize_text_soft(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = URL_RE.sub(" ", text)
    text = NEWLINE_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def tokenize(text: str):
    return WORD_RE.findall(text.lower())

def count_words_from_vocab(tokens, vocab):
    return sum(token in vocab for token in tokens)

def safe_div(a, b):
    return a / b if b else 0.0

def max_consecutive_char_repeat(text: str) -> int:
    max_repeat = 1
    current = 1
    for i in range(1, len(text)):
        if text[i] == text[i - 1]:
            current += 1
            max_repeat = max(max_repeat, current)
        else:
            current = 1
    return max_repeat if text else 0

# ---------------------------
# Блоки признаков
# ---------------------------

def extract_raw_features(text: str) -> dict:
    raw = text if isinstance(text, str) else ""
    words_raw = WORD_RE.findall(raw)
    sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(raw) if s.strip()]
    caps_words = CAPS_WORD_RE.findall(raw)

    num_chars = len(raw)
    num_words_raw = len(words_raw)
    num_sentences = len(sentences)

    num_uppercase_letters = len(UPPERCASE_LETTER_RE.findall(raw))
    num_exclaims = raw.count('!')
    num_questions = raw.count('?')
    num_dots = raw.count('.')
    num_commas = raw.count(',')
    num_colons = raw.count(':')
    num_semicolons = raw.count(';')
    num_quotes = sum(raw.count(ch) for ch in ['"', "'", '«', '»', '„', '“', '”'])
    num_brackets = sum(raw.count(ch) for ch in '()[]{}')
    num_digits = len(DIGIT_RE.findall(raw))
    num_spaces = raw.count(' ')
    num_newlines = len(re.findall(r'[\r\n]', raw))
    num_punct = sum(ch in PUNCT_CHARS for ch in raw)
    num_mixed_punct = len(MULTI_PUNCT_RE.findall(raw))
    num_repeated_chars = len(REPEAT_CHAR_RE.findall(raw))
    max_char_repeat = max_consecutive_char_repeat(raw)

    return {
        'num_chars': num_chars,
        'num_words_raw': num_words_raw,
        'num_sentences': num_sentences,
        'num_uppercase_letters': num_uppercase_letters,
        'uppercase_ratio': safe_div(num_uppercase_letters, num_chars),
        'num_exclaims': num_exclaims,
        'num_questions': num_questions,
        'num_dots': num_dots,
        'num_commas': num_commas,
        'num_colons': num_colons,
        'num_semicolons': num_semicolons,
        'num_quotes': num_quotes,
        'num_brackets': num_brackets,
        'num_digits': num_digits,
        'digit_ratio': safe_div(num_digits, num_chars),
        'num_punct': num_punct,
        'punct_ratio': safe_div(num_punct, num_chars),
        'num_spaces': num_spaces,
        'space_ratio': safe_div(num_spaces, num_chars),
        'num_newlines': num_newlines,
        'num_mixed_punct': num_mixed_punct,
        'has_many_exclaims': int(num_exclaims >= 3),
        'has_many_questions': int(num_questions >= 3),
        'has_ellipsis': int('...' in raw),
        'num_caps_words': len(caps_words),
        'has_caps_word': int(len(caps_words) > 0),
        'max_caps_word_len': max((len(w) for w in caps_words), default=0),
        'num_repeated_chars': num_repeated_chars,
        'max_char_repeat': max_char_repeat,
        'num_long_tokens_raw': sum(len(w) > 10 for w in words_raw),
    }

def extract_length_features(text: str) -> dict:
    soft = normalize_text_soft(text)
    tokens = WORD_RE.findall(soft)
    token_lengths = [len(t) for t in tokens]
    sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(soft) if s.strip()]
    sentence_word_lengths = [len(WORD_RE.findall(s)) for s in sentences]
    sentence_char_lengths = [len(s) for s in sentences]

    num_words = len(tokens)
    num_unique_words = len(set(t.lower() for t in tokens))

    return {
        'avg_word_len': float(np.mean(token_lengths)) if token_lengths else 0.0,
        'median_word_len': float(np.median(token_lengths)) if token_lengths else 0.0,
        'std_word_len': float(np.std(token_lengths)) if token_lengths else 0.0,
        'num_unique_words': num_unique_words,
        'unique_ratio': safe_div(num_unique_words, num_words),
        'num_short_words': sum(len(t) <= 2 for t in tokens),
        'num_medium_words': sum(3 <= len(t) <= 6 for t in tokens),
        'num_long_words': sum(len(t) >= 7 for t in tokens),
        'avg_sentence_len_words': float(np.mean(sentence_word_lengths)) if sentence_word_lengths else 0.0,
        'avg_sentence_len_chars': float(np.mean(sentence_char_lengths)) if sentence_char_lengths else 0.0,
        'num_empty_like_fragments': max(0, len(SENTENCE_SPLIT_RE.split(soft)) - len(sentences)),
    }

def extract_lexicon_features(text: str) -> dict:
    lower = normalize_text_soft(text).lower()
    tokens = tokenize(lower)

    pos_count = count_words_from_vocab(tokens, POS_WORDS)
    neg_count = count_words_from_vocab(tokens, NEG_WORDS)
    intensifier_count = count_words_from_vocab(tokens, INTENSIFIERS)
    negation_count = count_words_from_vocab(tokens, NEGATIONS)
    delivery_count = count_words_from_vocab(tokens, DELIVERY_WORDS)
    quality_count = count_words_from_vocab(tokens, QUALITY_WORDS)
    size_count = count_words_from_vocab(tokens, SIZE_WORDS)

    num_words = len(tokens)

    return {
        'num_negations': negation_count,
        'num_intensifiers': intensifier_count,
        'num_positive_words': pos_count,
        'num_negative_words': neg_count,
        'sentiment_lexicon_diff': pos_count - neg_count,
        'num_delivery_words': delivery_count,
        'num_quality_words': quality_count,
        'num_size_words': size_count,
        'neg_words_per_word': safe_div(neg_count, num_words),
        'pos_words_per_word': safe_div(pos_count, num_words),
        'lexicon_balance_ratio': safe_div(pos_count + 1, neg_count + 1),
    }

def extract_pattern_features(text: str) -> dict:
    lower = normalize_text_soft(text).lower()
    return {
        name: int(bool(re.search(pattern, lower)))
        for name, pattern in PATTERNS.items()
    }

def extract_ratio_features(features: dict) -> dict:
    num_words = features.get('num_words_raw', 0)
    num_sentences = features.get('num_sentences', 0)

    return {
        'exclaims_per_word': safe_div(features.get('num_exclaims', 0), num_words),
        'questions_per_word': safe_div(features.get('num_questions', 0), num_words),
        'caps_per_word': safe_div(features.get('num_caps_words', 0), num_words),
        'punct_per_word': safe_div(features.get('num_punct', 0), num_words),
        'repeat_chars_per_word': safe_div(features.get('num_repeated_chars', 0), num_words),
        'avg_punct_per_sentence': safe_div(features.get('num_punct', 0), num_sentences),
        'avg_exclaims_per_sentence': safe_div(features.get('num_exclaims', 0), num_sentences),
    }

# ---------------------------
# Главная функция
# ---------------------------

def extract_all_features(text: str) -> dict:
    features = {}
    features.update(extract_raw_features(text))
    features.update(extract_length_features(text))
    features.update(extract_lexicon_features(text))
    features.update(extract_pattern_features(text))
    features.update(extract_ratio_features(features))
    return features

# ---------------------------
# Применение к датасету
# -------------------------

features_df = df_wc['review'].apply(extract_all_features).apply(pd.Series)

print(features_df.shape)
features_df.head()


(90000, 70)


,num_chars,num_words_raw,num_sentences,num_uppercase_letters,uppercase_ratio,num_exclaims,num_questions,num_dots,num_commas,num_colons,...,has_size_issue,has_strong_negative_phrase,has_strong_positive_phrase,exclaims_per_word,questions_per_word,caps_per_word,punct_per_word,repeat_chars_per_word,avg_punct_per_sentence,avg_exclaims_per_sentence
0,172.0,24.0,2.0,6.0,0.034884,5.0,0.0,7.0,0.0,0.0,...,0.0,0.0,0.0,0.208333,0.0,0.041667,0.583333,0.083333,7.0,2.50
1,80.0,12.0,2.0,2.0,0.025000,0.0,0.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.250000,0.000000,1.5,0.00
2,204.0,30.0,4.0,4.0,0.019608,3.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.100000,0.0,0.000000,0.266667,0.000000,2.0,0.75
3,86.0,13.0,2.0,0.0,0.000000,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.153846,0.000000,1.0,0.00
4,45.0,6.0,1.0,1.0,0.022222,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.333333,0.000000,2.0,0.00


In [58]:
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}

df_model = df_wc.copy()
df_model['review'] = df_model['review'].astype(str)
df_model['target'] = df_model['sentiment'].map(label_map)

df_model = df_model.dropna(subset=['review', 'target']).reset_index(drop=True)

X_text = df_model['review']
y = df_model['target']


In [59]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [60]:
from collections import Counter
import re

STOPWORDS_RU = {
    'и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то',
    'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за',
    'бы', 'по', 'ее', 'мне', 'есть', 'они', 'тут', 'где', 'или', 'ни', 'мы',
    'это', 'от', 'до', 'для', 'из', 'под', 'над', 'при', 'без', 'после'
}

def simple_tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'[а-яёa-z]+', text.lower())

def get_top_words_by_class(texts, labels, top_n=30, min_len=3, stopwords=None):
    stopwords = stopwords or set()
    class_top_words = {}

    for cls in sorted(set(labels)):
        class_texts = [text for text, label in zip(texts, labels) if label == cls]
        all_tokens = []

        for text in class_texts:
            tokens = simple_tokenize(text)
            tokens = [t for t in tokens if len(t) >= min_len and t not in stopwords]
            all_tokens.extend(tokens)

        class_top_words[cls] = [word for word, _ in Counter(all_tokens).most_common(top_n)]

    return class_top_words


In [61]:
class_top_words = get_top_words_by_class(
    texts=X_train_text.tolist(),
    labels=y_train.tolist(),
    top_n=50,
    min_len=3,
    stopwords=STOPWORDS_RU
)

class_top_words


{0.0: ['деньги',
  'товар',
  'очень',
  'продавец',
  'спор',
  'размер',
  'вернули',
  'заказ',
  'качество',
  'пришёл',
  'пришел',
  'продавца',
  'открыла',
  'соответствует',
  'месяца',
  'получила',
  'только',
  'заказала',
  'пришла',
  'ткань',
  'платье',
  'фото',
  'даже',
  'нет',
  'денег',
  'больше',
  'заказывала',
  'вообще',
  'просто',
  'пришло',
  'ждала',
  'советую',
  'цвет',
  'вернул',
  'этого',
  'рекомендую',
  'ничего',
  'долго',
  'раз',
  'нитки',
  'синтетика',
  'носить',
  'товара',
  'уже',
  'было',
  'меня',
  'рукава',
  'итоге',
  'хотя',
  'если'],
 2.0: ['очень',
  'размер',
  'спасибо',
  'качество',
  'доставка',
  'хорошо',
  'быстро',
  'рекомендую',
  'рост',
  'супер',
  'довольна',
  'продавцу',
  'платье',
  'отлично',
  'быстрая',
  'хорошее',
  'продавец',
  'заказала',
  'ткань',
  'цвет',
  'недели',
  'товар',
  'немного',
  'продавца',
  'заказ',
  'раз',
  'соответствует',
  'идеально',
  'дней',
  'хорошая',
  'фото',
  'п

In [62]:
def extract_top_word_features(text, class_top_words):
    tokens = simple_tokenize(text)
    token_set = set(tokens)
    num_words = len(tokens)

    features = {}

    for cls, words in class_top_words.items():
        words_set = set(words)

        match_count = sum(token in words_set for token in tokens)
        unique_match_count = len(token_set & words_set)

        features[f'class_{cls}_top_match_count'] = match_count
        features[f'class_{cls}_top_unique_match_count'] = unique_match_count
        features[f'class_{cls}_top_match_ratio'] = match_count / num_words if num_words else 0.0
        features[f'class_{cls}_has_top_word'] = int(unique_match_count > 0)

    # Разница между позитивным и негативным словарём часто полезна
    if -1 in class_top_words and 1 in class_top_words:
        features['pos_neg_top_diff'] = (
            features.get('class_1_top_match_count', 0) -
            features.get('class_-1_top_match_count', 0)
        )

    return features

X_train_top = X_train_text.apply(
    lambda text: extract_top_word_features(text, class_top_words)
).apply(pd.Series)

X_test_top = X_test_text.apply(
    lambda text: extract_top_word_features(text, class_top_words)
).apply(pd.Series)

X_train_top.head()


,class_0.0_top_match_count,class_0.0_top_unique_match_count,class_0.0_top_match_ratio,class_0.0_has_top_word,class_2.0_top_match_count,class_2.0_top_unique_match_count,class_2.0_top_match_ratio,class_2.0_has_top_word
53515,1.0,1.0,0.166667,1.0,3.0,3.0,0.500000,1.0
4919,5.0,4.0,0.208333,1.0,7.0,6.0,0.291667,1.0
37785,2.0,1.0,0.153846,1.0,3.0,2.0,0.230769,1.0
8947,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0
24289,5.0,5.0,0.294118,1.0,3.0,3.0,0.176471,1.0


In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

word_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 3),
    min_df=7,
    max_df=0.95,
    max_features=8000,
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(X_train_text)
X_test_word = word_vectorizer.transform(X_test_text)

print('word tfidf:', X_train_word.shape, X_test_word.shape)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 7),
    min_df=7,
    max_df=0.95,
    max_features=5000,
    sublinear_tf=True
)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_test_char = char_vectorizer.transform(X_test_text)

print('char tfidf:', X_train_char.shape, X_test_char.shape)


word tfidf: (48000, 8000) (12000, 8000)
char tfidf: (48000, 5000) (12000, 5000)


In [64]:
from sklearn.decomposition import TruncatedSVD
import pandas as pd

svd_word = TruncatedSVD(n_components=200, random_state=42)
svd_char = TruncatedSVD(n_components=150, random_state=42)

# fit только на train, transform и для train, и для test
X_train_word_svd = svd_word.fit_transform(X_train_word)
X_test_word_svd = svd_word.transform(X_test_word)

X_train_char_svd = svd_char.fit_transform(X_train_char)
X_test_char_svd = svd_char.transform(X_test_char)

print('word svd:', X_train_word_svd.shape, X_test_word_svd.shape)
print('char svd:', X_train_char_svd.shape, X_test_char_svd.shape)


word svd: (48000, 200) (12000, 200)
char svd: (48000, 150) (12000, 150)


In [65]:
X_train_word_svd_df = pd.DataFrame(
    X_train_word_svd,
    index=X_train_text.index,
    columns=[f'word_svd_{i}' for i in range(X_train_word_svd.shape[1])]
)

X_test_word_svd_df = pd.DataFrame(
    X_test_word_svd,
    index=X_test_text.index,
    columns=[f'word_svd_{i}' for i in range(X_test_word_svd.shape[1])]
)

X_train_char_svd_df = pd.DataFrame(
    X_train_char_svd,
    index=X_train_text.index,
    columns=[f'char_svd_{i}' for i in range(X_train_char_svd.shape[1])]
)

X_test_char_svd_df = pd.DataFrame(
    X_test_char_svd,
    index=X_test_text.index,
    columns=[f'char_svd_{i}' for i in range(X_test_char_svd.shape[1])]
)


In [66]:
# Сбрасываем индексы, чтобы concat не склеивал по старым индексам криво
X_train_features = X_train_features.reset_index(drop=True)
X_test_features = X_test_features.reset_index(drop=True)

X_train_word_svd_df = X_train_word_svd_df.reset_index(drop=True)
X_test_word_svd_df = X_test_word_svd_df.reset_index(drop=True)

X_train_char_svd_df = X_train_char_svd_df.reset_index(drop=True)
X_test_char_svd_df = X_test_char_svd_df.reset_index(drop=True)

X_train_final = pd.concat(
    [
        X_train_features,
        X_train_word_svd_df,
        X_train_char_svd_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test_features,
        X_test_word_svd_df,
        X_test_char_svd_df
    ],
    axis=1
)

print(X_train_final.shape)
print(X_test_final.shape)
X_train_final.head()


NameError: name 'X_train_features' is not defined